In [5]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF, ChiSqSelector, Normalizer, StringIndexer
from pyspark.ml.classification import LinearSVC, OneVsRest
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

In [6]:
spark = SparkSession.builder \
    .appName("Text Classification with SVM") \
    .getOrCreate()

In [7]:
data_path = "hdfs:///user/dic24_shared/amazon-reviews/full/reviews_devset.json"
data = spark.read.json(data_path)

In [8]:
data = data.select("reviewText", "category").na.drop()

In [9]:
indexer = StringIndexer(inputCol="category", outputCol="label")

In [10]:
pattern = r'\s+|\d+|[()\[\]{}.,;:!?\+\-=_\"\'`~#@&*%€$§\\\/]+'
regex_tokenizer = RegexTokenizer(inputCol="reviewText", outputCol="words", pattern=pattern, gaps=True,minTokenLength =2)

In [11]:
stop_words_remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")

In [12]:
vectorizer = CountVectorizer(inputCol="filtered_words", outputCol="vectorized_Features")

In [13]:
idf = IDF(inputCol="vectorized_Features", outputCol="features")

In [14]:
chi_selector = ChiSqSelector(numTopFeatures=2000, featuresCol="features", outputCol="selectedFeatures", labelCol="label")

In [15]:
pipeline = Pipeline(stages=[indexer, regex_tokenizer, stop_words_remover, vectorizer, idf, chi_selector])

In [16]:
fitted_pipeline = pipeline.fit(data)

In [17]:
trasnsformed_data = fitted_pipeline.transform(data)

In [18]:
data.unpersist()

DataFrame[reviewText: string, category: string]

In [19]:
chi_selector_model = fitted_pipeline.stages[5]
selected_indices = chi_selector_model.selectedFeatures

vocabulary = fitted_pipeline.stages[3].vocabulary

selected_words = [vocabulary[i] for i in selected_indices]

sorted_words = sorted(selected_words)

words_line = " ".join(sorted_words)

with open("output_ds.txt", "w") as file:
    file.write(words_line)

In [20]:
spark.stop()